# 📊 UNIVERSAL EVALUATION TOOLKIT CHO BÀI TOÁN DOCUMENT DEWARPING

Notebook này được thiết kế theo kiến trúc **Model-Agnostic (Dùng chung cho MỌI mô hình)**:
* Hoạt động với bất kỳ mô hình nắn phẳng tài liệu nào: **DocTr, DewarpNet, DocUNet, FDRNet, Paper-Dewarp, hoặc Model bạn tự nghiên cứu/cải tiến**.
* Đọc ảnh nắn phẳng từ bất kỳ thư mục nào và tự động so khớp với ảnh scan Ground Truth của bộ benchmark.
* **Các chỉ số đánh giá tiêu chuẩn:**
  * **MS-SSIM (↑):** Độ tương đồng cấu trúc thị giác (Multi-Scale Structural Similarity).
  * **CER (↓):** Tỷ lệ lỗi nhận dạng ký tự (Character Error Rate).
  * **Levenshtein ED (↓):** Số ký tự sai cần chỉnh sửa (Edit Distance).
  * **Thời gian xử lý trung bình (s/ảnh)**.
* Tự động lưu bảng kết quả chi tiết từng ảnh và dòng tổng kết vào file CSV trên Google Drive để phân tích bằng Excel.

### Bước 1: Kết nối Google Drive & Cài đặt thư viện đánh giá

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q rapidocr-onnxruntime rapidfuzz scikit-image opencv-python pillow pandas

### Bước 2: Tải bộ công cụ đo kiểm (Từ GitHub hoặc file cục bộ)

In [ ]:
import os
# Đổi sang URL GitHub repo của bạn
GITHUB_REPO_URL = "https://github.com/nck345/test_doctr.git"

if not os.path.exists('/content/eval_toolkit'):
    !git clone {GITHUB_REPO_URL} /content/eval_toolkit

%cd /content/eval_toolkit

### Bước 3: Cấu hình thư mục mô hình cần đánh giá
*(Bạn chỉ cần đổi `MODEL_NAME` và `PRED_DIR` tương ứng với mô hình bạn muốn đo)*

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/Document_Dewarp_Benchmark"

# 1. Tên mô hình đang đánh giá (Ví dụ: 'DocTr', 'DewarpNet', 'DocUNet', 'My_Improvement')
MODEL_NAME = "DocTr"

# 2. Thư mục chứa ảnh nắn phẳng được xuất ra từ mô hình này
PRED_DIR = f"{DRIVE_DIR}/predictions/{MODEL_NAME}"

# 3. Thư mục chứa ảnh scan chuẩn Ground Truth (DocUNet benchmark)
GT_DIR = f"{DRIVE_DIR}/data_benchmark/scan/scan"

# 4. Nơi lưu file báo cáo kết quả
SAVE_CSV = f"{DRIVE_DIR}/eval_reports/{MODEL_NAME}_metrics.csv"

print("Model Name  :", MODEL_NAME)
print("Predictions :", PRED_DIR, "- Tồn tại:", os.path.exists(PRED_DIR))
print("Ground Truth:", GT_DIR, "- Tồn tại:", os.path.exists(GT_DIR))
print("Save CSV    :", SAVE_CSV)

### Bước 4: Chạy đo kiểm toàn diện
Chạy đo tự động từng ảnh và xuất bảng báo cáo chi tiết.

In [ ]:
!python eval_benchmark.py --pred_dir "{PRED_DIR}" --gt_dir "{GT_DIR}" --model_name "{MODEL_NAME}" --save_csv "{SAVE_CSV}"

### Bước 5: Xem nhanh kết quả tổng hợp bằng Pandas

In [ ]:
import pandas as pd
if os.path.exists(SAVE_CSV):
    df = pd.read_csv(SAVE_CSV)
    print("\n--- 5 DÒNG KẾT QUẢ ĐẦU TIÊN ---")
    display(df.head())
    print("\n--- DÒNG TỔNG HỢP TRUNG BÌNH TOÀN TẬP ---")
    display(df.tail(2))
else:
    print("Chưa tìm thấy file CSV báo cáo.")